In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, math, random, zipfile, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF

from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

# =========================
# Config
# =========================
ZIP_PATH = "/content/drive/MyDrive/EdgeCard_System/dataset_rec.zip"
DATA_DIR = "/content/dataset_rec"
OUTPUT_DIR = "/content/drive/MyDrive/EdgeCard_System/stage_3/mobilenetv3_crnn"

IMG_W = 320
EVAL_H = 48
TRAIN_HEIGHTS = [32, 48, 64]
MAX_TEXT_LENGTH = 40

BATCH_SIZE = 128
EPOCHS = 50
BASE_LR = 5e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
SEED = 1024

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))

In [ ]:
def prepare_label(src_name, dst_name):
    src = os.path.join(DATA_DIR, src_name)
    dst = os.path.join(DATA_DIR, dst_name)

    lines, chars, max_len = [], set(), 0

    with open(src, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\r\n")
            if not line:
                continue

            path, text = line.split("\t", 1)
            path = path.replace("\\", "/")
            text = unicodedata.normalize("NFC", text)

            lines.append(f"{path}\t{text}")
            chars.update(text)
            max_len = max(max_len, len(text))

    with open(dst, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    return chars, max_len, len(lines)


splits = {}
for split in ["train", "valid", "test"]:
    splits[split] = prepare_label(
        f"{split}_label.txt",
        f"{split}_crnn.txt"
    )

vietnamese_chars = (
    "aAàÀảẢãÃáÁạẠăĂằẰẳẲẵẴắẮặẶâÂầẦẩẨẫẪấẤậẬ"
    "bBcCdDđĐeEèÈẻẺẽẼéÉẹẸêÊềỀểỂễỄếẾệỆ"
    "fFgGhHiIìÌỉỈĩĨíÍịỊjJkKlLmMnNoO"
    "òÒỏỎõÕóÓọỌôÔồỒổỔỗỖốỐộỘơƠờỜởỞỡỠớỚợỢ"
    "pPqQrRsStTuUùÙủỦũŨúÚụỤưƯừỪửỬữỮứỨựỰ"
    "vVwWxXyYỳỲỷỶỹỸýÝỵỴzZ"
    "0123456789"
    "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"
)

characters = list(dict.fromkeys(vietnamese_chars)) + [" "]

BLANK_IDX = 0
char2idx = {c: i + 1 for i, c in enumerate(characters)}
idx2char = {i + 1: c for i, c in enumerate(characters)}
NUM_CLASSES = len(characters) + 1

all_chars = set().union(*(v[0] for v in splits.values()))
max_len = max(v[1] for v in splits.values())

assert not (all_chars - set(characters)), f"Missing chars: {all_chars - set(characters)}"
assert max_len <= MAX_TEXT_LENGTH, f"Label dài nhất = {max_len}"

print(f"Train: {splits['train'][2]} | Valid: {splits['valid'][2]} | Test: {splits['test'][2]}")
print(f"Classes: {NUM_CLASSES} | Max label: {max_len}")

In [ ]:
class MobileNetV3OCRBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        base = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
        self.features = nn.Sequential(*list(base.features.children())[:-1])

        stride2 = [
            m for m in self.features.modules()
            if isinstance(m, nn.Conv2d) and m.stride == (2, 2)
        ]

        assert len(stride2) >= 5
        stride2[2].stride = (2, 1)
        stride2[3].stride = (2, 1)
        stride2[4].stride = (1, 1)

    def forward(self, x):
        return self.features(x)


class TemporalConvBlock(nn.Module):
    def __init__(self, dim=256, dropout=0.1):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(dim, dim, 3, padding=1, groups=dim, bias=False),
            nn.Conv1d(dim, dim, 1, bias=False),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x
        x = self.conv(x.transpose(1, 2)).transpose(1, 2)
        return self.norm(x + residual)


class MobileNetV3CRNN(nn.Module):
    def __init__(self, num_classes, hidden_size=256, lstm_layers=2, dropout=0.1):
        super().__init__()

        self.backbone = MobileNetV3OCRBackbone()

        self.channel_proj = nn.Sequential(
            nn.Conv2d(160, 256, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.Hardswish()
        )

        self.spatial_pool = nn.AdaptiveAvgPool2d((2, 80))

        self.sequence_proj = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.temporal = TemporalConvBlock(256, dropout)

        self.rnn = nn.LSTM(
            input_size=256,
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0
        )

        self.rnn_norm = nn.LayerNorm(hidden_size * 2)
        self.head = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        x = self.channel_proj(self.backbone(x))
        x = self.spatial_pool(x)
        x = x.permute(0, 3, 1, 2).flatten(2)
        x = self.sequence_proj(x)
        x = self.temporal(x)
        x, _ = self.rnn(x)
        return self.head(self.rnn_norm(x))


model = MobileNetV3CRNN(NUM_CLASSES).to(DEVICE)

# Một sanity-check quan trọng là đủ
model.eval()
with torch.no_grad():
    for h in TRAIN_HEIGHTS:
        y = model(torch.randn(2, 3, h, IMG_W, device=DEVICE))
        assert y.shape == (2, 80, NUM_CLASSES)

params = sum(p.numel() for p in model.parameters())

print(f"Model : MobileNetV3-CRNN")
print(f"Params: {params / 1e6:.3f} M")
print(f"FP32  : {params * 4 / 1024**2:.2f} MB")

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class OCRDataset(torch.utils.data.Dataset):
    def __init__(self, label_file):
        with open(label_file, encoding="utf-8") as f:
            self.samples = [
                line.rstrip("\r\n").split("\t", 1)
                for line in f if line.strip()
            ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, text = self.samples[idx]
        image = Image.open(os.path.join(DATA_DIR, path)).convert("RGB")
        return image, text


def augment_image(img):
    if random.random() < 0.3:
        img = img.rotate(
            random.uniform(-3, 3),
            resample=Image.Resampling.BILINEAR,
            fillcolor=(255, 255, 255)
        )

    if random.random() < 0.3:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.85, 1.15))

    if random.random() < 0.3:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.15))

    if random.random() < 0.15:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.1, 0.7)))

    return img


def resize_pad(image, target_h, target_w=IMG_W):
    w, h = image.size
    new_w = max(1, min(target_w, round(w * target_h / h)))

    image = image.resize((new_w, target_h), Image.Resampling.BILINEAR)

    canvas = Image.new("RGB", (target_w, target_h), (255, 255, 255))
    canvas.paste(image, (0, 0))

    x = TF.to_tensor(canvas)
    return TF.normalize(x, IMAGENET_MEAN, IMAGENET_STD)


def encode_text(text):
    return torch.tensor([char2idx[c] for c in text], dtype=torch.long)


class OCRCollate:
    def __init__(self, train):
        self.train = train

    def __call__(self, batch):
        h = random.choice(TRAIN_HEIGHTS) if self.train else EVAL_H

        images, targets, lengths, texts = [], [], [], []

        for image, text in batch:
            if self.train:
                image = augment_image(image)

            target = encode_text(text)

            images.append(resize_pad(image, h))
            targets.append(target)
            lengths.append(len(target))
            texts.append(text)

        return {
            "images": torch.stack(images),
            "targets": torch.cat(targets),
            "target_lengths": torch.tensor(lengths, dtype=torch.long),
            "texts": texts,
            "height": h
        }


datasets = {
    split: OCRDataset(os.path.join(DATA_DIR, f"{split}_crnn.txt"))
    for split in ["train", "valid", "test"]
}

train_loader = torch.utils.data.DataLoader(
    datasets["train"], batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=True, drop_last=True,
    collate_fn=OCRCollate(True)
)

valid_loader = torch.utils.data.DataLoader(
    datasets["valid"], batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True, collate_fn=OCRCollate(False)
)

test_loader = torch.utils.data.DataLoader(
    datasets["test"], batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True, collate_fn=OCRCollate(False)
)

print(
    f"Train: {len(datasets['train'])} ({len(train_loader)} batches) | "
    f"Valid: {len(datasets['valid'])} ({len(valid_loader)}) | "
    f"Test: {len(datasets['test'])} ({len(test_loader)})"
)

In [ ]:
def ctc_decode(logits):
    results = []

    for seq in logits.argmax(-1):
        text, prev = [], -1

        for idx in seq.tolist():
            if idx != BLANK_IDX and idx != prev:
                text.append(idx2char[idx])
            prev = idx

        results.append("".join(text))

    return results


def edit_distance(a, b):
    prev = list(range(len(b) + 1))

    for i, ca in enumerate(a, 1):
        curr = [i]

        for j, cb in enumerate(b, 1):
            curr.append(min(
                curr[-1] + 1,
                prev[j] + 1,
                prev[j - 1] + (ca != cb)
            ))

        prev = curr

    return prev[-1]


def compute_metrics(preds, gts):
    exact = edits = chars = 0
    ned = 0.0

    for pred, gt in zip(preds, gts):
        exact += pred == gt
        d = edit_distance(pred, gt)

        edits += d
        chars += len(gt)
        ned += 1 - d / max(len(pred), len(gt), 1)

    n = len(gts)

    return {
        "accuracy": exact / n,
        "cer": edits / max(chars, 1),
        "ned": ned / n
    }

In [ ]:
criterion = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    betas=(0.9, 0.999),
    weight_decay=WEIGHT_DECAY
)

TOTAL_STEPS = EPOCHS * len(train_loader)
WARMUP_STEPS = WARMUP_EPOCHS * len(train_loader)


def lr_lambda(step):
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS

    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

USE_AMP = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

BEST_PATH = os.path.join(OUTPUT_DIR, "best.pth")
LAST_PATH = os.path.join(OUTPUT_DIR, "last.pth")
HISTORY_PATH = os.path.join(OUTPUT_DIR, "history.csv")


def save_best(epoch, best_acc):
    # Best chỉ cần weights để evaluation/deployment
    torch.save({
        "epoch": epoch,
        "best_acc": best_acc,
        "model_state_dict": model.state_dict()
    }, BEST_PATH)


def save_last(epoch, best_acc):
    # Last lưu full state để resume
    torch.save({
        "epoch": epoch,
        "best_acc": best_acc,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict()
    }, LAST_PATH)


def resume_training():
    ckpt = torch.load(LAST_PATH, map_location=DEVICE, weights_only=False)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])

    return ckpt["epoch"] + 1, ckpt["best_acc"]


print(f"LR: {BASE_LR} | Warmup: {WARMUP_EPOCHS} | AMP: {USE_AMP}")

In [ ]:
def ctc_loss(logits, targets, target_lengths):
    log_probs = logits.log_softmax(-1).permute(1, 0, 2)
    input_lengths = torch.full(
        (logits.size(0),),
        logits.size(1),
        dtype=torch.long
    )
    return criterion(log_probs, targets, input_lengths, target_lengths)


def train_one_epoch():
    model.train()

    total_loss = total_samples = 0
    pbar = tqdm(train_loader, desc="Train", leave=False)

    for batch in pbar:
        images = batch["images"].to(DEVICE, non_blocking=True)
        targets = batch["targets"].to(DEVICE, non_blocking=True)
        target_lengths = batch["target_lengths"]
        bs = images.size(0)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            loss = ctc_loss(model(images), targets, target_lengths)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()

        # Không step scheduler nếu AMP vừa skip optimizer step
        if scaler.get_scale() >= scale_before:
            scheduler.step()

        total_loss += loss.item() * bs
        total_samples += bs

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            h=batch["height"]
        )

    return total_loss / total_samples


@torch.no_grad()
def evaluate(loader, desc="Eval"):
    model.eval()

    total_loss = total_samples = 0
    preds, gts = [], []

    for batch in tqdm(loader, desc=desc, leave=False):
        images = batch["images"].to(DEVICE, non_blocking=True)
        targets = batch["targets"].to(DEVICE, non_blocking=True)
        target_lengths = batch["target_lengths"]
        bs = images.size(0)

        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
            loss = ctc_loss(logits, targets, target_lengths)

        total_loss += loss.item() * bs
        total_samples += bs

        preds.extend(ctc_decode(logits))
        gts.extend(batch["texts"])

    return total_loss / total_samples, compute_metrics(preds, gts), preds, gts

In [ ]:
RESUME = False

if RESUME and os.path.exists(LAST_PATH):
    START_EPOCH, BEST_ACC = resume_training()
    history = (
        pd.read_csv(HISTORY_PATH).to_dict("records")
        if os.path.exists(HISTORY_PATH) else []
    )
    print(f"Resume epoch {START_EPOCH + 1}, best={BEST_ACC:.4f}")
else:
    START_EPOCH, BEST_ACC, history = 0, 0.0, []


for epoch in range(START_EPOCH, EPOCHS):
    print(f"\n{'=' * 60}\nEpoch {epoch + 1}/{EPOCHS}\n{'=' * 60}")

    train_loss = train_one_epoch()
    val_loss, m, preds, gts = evaluate(valid_loader, "Valid")

    acc, cer, ned = m["accuracy"], m["cer"], m["ned"]
    lr = optimizer.param_groups[0]["lr"]

    print(
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}\n"
        f"Acc: {acc:.4f} | CER: {cer:.4f} | NED: {ned:.4f} | LR: {lr:.2e}"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": acc,
        "val_cer": cer,
        "val_ned": ned,
        "lr": lr
    })

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    if acc > BEST_ACC:
        BEST_ACC = acc
        save_best(epoch, BEST_ACC)
        print(f"✅ New best: {BEST_ACC:.4f}")

    save_last(epoch, BEST_ACC)

    # Chỉ in sample khi best mới hoặc mỗi 5 epoch
    if acc == BEST_ACC or (epoch + 1) % 5 == 0:
        for gt, pred in list(zip(gts, preds))[:3]:
            print(f"GT  : {gt}")
            print(f"Pred: {pred}")
            print("-" * 30)


print(f"\nTraining finished | Best Val Acc: {BEST_ACC:.4f}")

In [ ]:
ckpt = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])

test_loss, test_metrics, test_preds, test_gts = evaluate(
    test_loader,
    desc="Test"
)

print("=" * 60)
print("MOBILENETV3-CRNN TEST RESULT")
print("=" * 60)
print(f"Best epoch : {ckpt['epoch'] + 1}")
print(f"Test Loss  : {test_loss:.4f}")
print(f"Exact Acc  : {test_metrics['accuracy']:.4f} ({test_metrics['accuracy']*100:.2f}%)")
print(f"CER        : {test_metrics['cer']:.4f} ({test_metrics['cer']*100:.2f}%)")
print(f"NED        : {test_metrics['ned']:.4f} ({test_metrics['ned']*100:.2f}%)")
print(f"Samples    : {len(test_gts)}")

wrong = [
    (gt, pred)
    for gt, pred in zip(test_gts, test_preds)
    if gt != pred
]

print(f"Correct    : {len(test_gts) - len(wrong)}")
print(f"Wrong      : {len(wrong)}")

print("\nMột số lỗi:")
for gt, pred in wrong[:10]:
    print(f"GT  : {gt}")
    print(f"Pred: {pred}")
    print("-" * 40)

In [ ]:
pd.DataFrame({
    "ground_truth": test_gts,
    "prediction": test_preds,
    "correct": [g == p for g, p in zip(test_gts, test_preds)]
}).to_csv(
    os.path.join(OUTPUT_DIR, "test_predictions.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("Saved test_predictions.csv")